# 01 - Transformar datos para agregados de negocio
Ejecuta las celdas de transformación para crear salidas agregadas para reportes.

#### Celda - Configuración de sesión Spark
Esta celda habilita dos características de Fabric que optimizan cómo se escriben y leen los datos en las celdas posteriores. V-order optimiza el diseño de parquet para lecturas más rápidas y mejor compresión. Optimize Write reduce el número de archivos escritos y aumenta el tamaño de los archivos individuales.

In [ ]:
# Configuración de Spark para optimizar escritura y lectura en parquet/delta
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

#### Celda - Cargar tablas de origen
Esta celda carga las tablas Delta de origen necesarias para las transformaciones de agregados de negocio.

In [ ]:
# Carga de tablas Delta de origen para agregaciones de negocio
df_fact_sale = spark.read.format("delta").load("Tables/silver/fact_sale")
df_dimension_date = spark.read.format("delta").load("Tables/silver/dimension_date")
df_dimension_city = spark.read.format("delta").load("Tables/silver/dimension_city")
df_dimension_employee = spark.read.format("delta").load("Tables/silver/dimension_employee")

#### Celda - Agregar ventas por fecha y ciudad
Esta celda calcula los totales mensuales de ventas por ciudad y escribe el resultado en `aggregate_sale_by_date_city`.

In [ ]:
sale_by_date_city = (
    df_fact_sale.alias("sale")
    .join(df_dimension_date.alias("date"), df_fact_sale.InvoiceDateKey == df_dimension_date.Date, "inner")
    .join(df_dimension_city.alias("city"), df_fact_sale.CityKey == df_dimension_city.CityKey, "inner")
    .select("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "city.City", "city.StateProvince", "city.SalesTerritory", "sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .groupBy("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "city.City", "city.StateProvince", "city.SalesTerritory")
    .sum("sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .withColumnRenamed("sum(TotalExcludingTax)", "SumOfTotalExcludingTax")
    .withColumnRenamed("sum(TaxAmount)", "SumOfTaxAmount")
    .withColumnRenamed("sum(TotalIncludingTax)", "SumOfTotalIncludingTax")
    .withColumnRenamed("sum(Profit)", "SumOfProfit")
    .orderBy("date.Date", "city.StateProvince", "city.City")
)

sale_by_date_city.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/gold/aggregate_sale_by_date_city")

#### Celda - Agregar ventas por fecha y empleado
Esta celda calcula los totales mensuales de ventas por empleado y escribe el resultado en `aggregate_sale_by_date_employee`.

In [ ]:
sale_by_date_employee = (
    df_fact_sale.alias("sale")
    .join(df_dimension_date.alias("date"), df_fact_sale.InvoiceDateKey == df_dimension_date.Date, "inner")
    .join(df_dimension_employee.alias("employee"), df_fact_sale.SalespersonKey == df_dimension_employee.EmployeeKey, "inner")
    .select("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "employee.PreferredName", "employee.Employee", "sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .groupBy("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "employee.PreferredName", "employee.Employee")
    .sum("sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .withColumnRenamed("sum(TotalExcludingTax)", "SumOfTotalExcludingTax")
    .withColumnRenamed("sum(TaxAmount)", "SumOfTaxAmount")
    .withColumnRenamed("sum(TotalIncludingTax)", "SumOfTotalIncludingTax")
    .withColumnRenamed("sum(Profit)", "SumOfProfit")
    .orderBy("date.Date", "employee.PreferredName", "employee.Employee")
)

sale_by_date_employee.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/gold/aggregate_sale_by_date_employee")